# SVM HIV Classification with 5-Fold CV

This notebook evaluates the SVM model on the HIV dataset using Morgan fingerprints, stratified 5-fold cross-validation, fold-level validation/test splits, and summary statistics (mean, std, variance).

In [ ]:

# If needed in Colab:
!pip -q install rdkit

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler


In [2]:

SEED = 42
N_SPLITS = 5
VAL_SIZE_WITHIN_TEMP = 0.5  # temp set is split equally into val and test
RADIUS = 2
N_BITS = 2048

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def smiles_to_morgan_fp(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp, dtype=np.float32))
        else:
            fps.append(np.zeros(n_bits, dtype=np.float32))
    return np.array(fps, dtype=np.float32)

def build_model(random_state):
    from sklearn.svm import SVC
    return SVC(
        kernel='rbf',
        probability=True,
        random_state=random_state
    )

def maybe_scale(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test


def evaluate_fold(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }, y_score


In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X = smiles_to_morgan_fp(df["smiles"], radius=RADIUS, n_bits=N_BITS)
y = df["HIV_active"].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results = []
roc_curves = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold} / {N_SPLITS} =====")
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    metrics, y_score = evaluate_fold(model, X_test, y_test)
    fold_results.append({
        "fold": fold,
        "test_accuracy": metrics["accuracy"],
        "test_precision": metrics["precision"],
        "test_recall": metrics["recall"],
        "test_f1": metrics["f1"],
        "test_roc_auc": metrics["roc_auc"]
    })

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_curves.append((fold, fpr, tpr, metrics["roc_auc"]))

    print({k: round(v, 4) if isinstance(v, float) else v for k, v in fold_results[-1].items()})

results_df = pd.DataFrame(fold_results)
results_df


In [ ]:

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:

plt.figure(figsize=(8, 6))
for fold, fpr, tpr, auc_val in roc_curves:
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC={auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM ROC Curves Across 5 Folds")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

results_df.to_csv("SVM_fold_results.csv", index=False)
summary_df.to_csv("SVM_summary_results.csv", index=False)

print("Saved:")
print("- SVM_fold_results.csv")
print("- SVM_summary_results.csv")


In [ ]:

# =========================================================
# FINAL CANDIDATE RANKING + DOCKING PREPARATION FOR SVM
# This cell uses out-of-fold predictions to avoid score leakage.
# =========================================================

from rdkit.Chem import Descriptors, Lipinski, Crippen, QED, Draw

# ---------- 1) Out-of-fold prediction export ----------
oof_records = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]
    smiles_temp = df.iloc[temp_idx]["smiles"].values

    X_val, X_test, y_val, y_test, smiles_val, smiles_test = train_test_split(
        X_temp, y_temp, smiles_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    fold_df = pd.DataFrame({
        "fold": fold,
        "smiles": smiles_test,
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob
    })
    oof_records.append(fold_df)

predictions_df = pd.concat(oof_records, ignore_index=True)
predictions_df.to_csv("SVM_oof_predictions.csv", index=False)

print("Saved: SVM_oof_predictions.csv")
display(predictions_df.head())


# ---------- 2) Candidate ranking ----------
# Keep unique molecules by the highest out-of-fold probability.
ranked_df = (
    predictions_df.sort_values("y_prob", ascending=False)
    .drop_duplicates(subset=["smiles"], keep="first")
    .reset_index(drop=True)
)

# Optional: prioritize molecules predicted as active
ranked_active_df = ranked_df[ranked_df["y_pred"] == 1].copy()

top_n = 50
top_candidates_df = ranked_active_df.head(top_n).copy()
top_candidates_df.to_csv("SVM_top_50_candidates.csv", index=False)

print(f"Saved: SVM_top_50_candidates.csv  |  Top rows: {len(top_candidates_df)}")
display(top_candidates_df.head(10))


# ---------- 3) RDKit descriptor calculation ----------
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "QED": QED.qed(mol)
    }

descriptor_rows = []
for _, row in top_candidates_df.iterrows():
    desc = compute_descriptors(row["smiles"])
    if desc is not None:
        desc.update({
            "smiles": row["smiles"],
            "fold": row["fold"],
            "y_true": row["y_true"],
            "y_pred": row["y_pred"],
            "y_prob": row["y_prob"]
        })
        descriptor_rows.append(desc)

descriptors_df = pd.DataFrame(descriptor_rows)
descriptors_df.to_csv("SVM_top_50_descriptors.csv", index=False)

print("Saved: SVM_top_50_descriptors.csv")
display(descriptors_df.head())


# ---------- 4) Lipinski filtering ----------
def lipinski_pass(row):
    return (
        row["MW"] <= 500 and
        row["LogP"] <= 5 and
        row["HBD"] <= 5 and
        row["HBA"] <= 10
    )

if not descriptors_df.empty:
    descriptors_df["Lipinski_pass"] = descriptors_df.apply(lipinski_pass, axis=1)
else:
    descriptors_df["Lipinski_pass"] = []

filtered_candidates_df = descriptors_df[descriptors_df["Lipinski_pass"] == True].copy()
filtered_candidates_df = filtered_candidates_df.sort_values("y_prob", ascending=False).reset_index(drop=True)

filtered_candidates_df.to_csv("SVM_docking_candidates_filtered.csv", index=False)
filtered_candidates_df[["smiles"]].to_csv("SVM_docking_input.smi", index=False, header=False)

print("Saved: SVM_docking_candidates_filtered.csv")
print("Saved: SVM_docking_input.smi")
display(filtered_candidates_df.head(10))


# ---------- 5) Molecule visualization ----------
top_vis = filtered_candidates_df.head(20).copy()
mols = [Chem.MolFromSmiles(sm) for sm in top_vis["smiles"] if Chem.MolFromSmiles(sm) is not None]

if len(mols) > 0:
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=4,
        subImgSize=(250, 250)
    )
    display(img)
else:
    print("No valid molecules found for visualization.")


# ---------- 6) Quick summary ----------
print("\nSummary")
print("-------")
print(f"Total OOF predictions          : {len(predictions_df)}")
print(f"Unique ranked molecules        : {len(ranked_df)}")
print(f"Top active candidates exported : {len(top_candidates_df)}")
print(f"Lipinski-passing candidates    : {len(filtered_candidates_df)}")


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# Your selected best 2 molecules
best_smiles = [
    "CCNc1ccccc1S(=O)(=O)c1ccccc1[N+](=O)[O-]",
    "O=[N+]([O-])c1ccccc1S(=O)(=O)c1cc(Cl)cc2c1NCCC2"
]

mols = [Chem.MolFromSmiles(sm) for sm in best_smiles]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(300,300),
    legends=["SVM - Molecule 1", "SVM - Molecule 2"]
)

display(img)